# 🏗️ Finnish Job Scraper v5 – Architect / BIM

| Site | Method |
|---|---|
| **Duunitori** | Playwright async (blocks plain requests with 403) |
| **Työmarkkinatori** | Playwright async (React SPA, needs JS rendering) |
| **Jobly** | `requests` + BeautifulSoup |
| **SAFA** | `requests` + BeautifulSoup |

**Run cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9**

## Cell 1 — Environment & Google Drive

In [1]:
#@title <font color=#1B7192> 1. Environment & Google Drive – Click to Run </font>  { display-mode: "form" }

import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/gdrive")

Running in: Google Colab
Mounted at /content/gdrive


## Cell 2 — Install dependencies
> `nest_asyncio` lets Playwright's async API run inside Colab's existing event loop.  
> `playwright install chromium` downloads a bundled browser — no ChromeDriver needed.

In [3]:
#@title <font color=#1B7192> 2. Install – Click to Run </font>  { display-mode: "form" }
if IN_COLAB:
  print("Installing packages...")
  !pip install -q requests beautifulsoup4 pandas openpyxl playwright nest_asyncio
  print("Downloading Playwright's bundled Chromium (≈150 MB, one-time)...")
  !playwright install chromium
  !apt-get update -qq && apt-get install -y libatk-bridge2.0-0 libatk1.0-0 libxcomposite1
  !ldconfig # Explicitly update the dynamic linker cache
  print("\n✅ Done!")
else:
    print("Local: pip install requests beautifulsoup4 pandas openpyxl playwright nest_asyncio")
    print("Also install ChromeDriver matching your Chrome version.")

Installing packages...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libatk-bridge2.0-0 is already the newest version (2.38.0-3).
libatk1.0-0 is already the newest version (2.36.0-3build1).
libxcomposite1 is already the newest version (1:0.4.5-1build2).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.
/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/li

In [4]:
print("Checking for libatk libraries:")
!ldconfig -p | grep libatk

Checking for libatk libraries:
	libatk-1.0.so.0 (libc6,x86-64) => /lib/x86_64-linux-gnu/libatk-1.0.so.0
	libatk-bridge-2.0.so.0 (libc6,x86-64) => /lib/x86_64-linux-gnu/libatk-bridge-2.0.so.0


In [5]:
print("Checking for libXcomposite library:")
!ldconfig -p | grep libXcomposite

Checking for libXcomposite library:
	libXcomposite.so.1 (libc6,x86-64) => /lib/x86_64-linux-gnu/libXcomposite.so.1


## Cell 3 — Imports, config & helpers
> **Re-run this cell to reset `all_jobs` before a fresh scrape.**

In [6]:
#@title <font color=#1B7192> 3. Imports & config – Click to Run </font>  { display-mode: "form" }

####################################################
#################### USER INPUT ####################
####################################################

Keywords  = "arkkitehti, projektiarkkitehti, BIM, architect"    #@param {type:"string"}
# Convert comma‑separated string → clean Python list
KEYWORDS = [kw.strip() for kw in Keywords.split(",") if kw.strip()]

####################################################
############### FUNCTION DEFINITIONS ###############
####################################################

import asyncio, re, time, requests, pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from IPython.display import display
import nest_asyncio

# ── KEY FIX: patch the running Colab event loop so async code can nest inside it
nest_asyncio.apply()

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "fi-FI,fi;q=0.9,en;q=0.8",
}

BROWSER_ARGS = [
    "--no-sandbox",
    "--disable-dev-shm-usage",
    "--disable-gpu",
]

def get_html(url, params=None, retries=3, delay=2):
    """Plain requests GET — for sites that don't need JS rendering."""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=15)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            print(f"  ⚠️  Attempt {attempt+1}/{retries} failed: {e}")
            time.sleep(delay)
    return None

all_jobs = []   # reset every time Cell 3 is run
print("✅ Setup complete. KEYWORDS:", KEYWORDS)

✅ Setup complete. KEYWORDS: ['arkkitehti', 'projektiarkkitehti', 'BIM', 'architect']


## Cell 4 — Scraper: Duunitori
> Uses `async_playwright` (Colab-compatible).  
> `asyncio.get_event_loop().run_until_complete(...)` runs the async function  
> inside Colab's existing loop — this is what `nest_asyncio` enables.

> Search URL: `https://duunitori.fi/tyopaikat?haku=arkkitehti`  
> Job URL: `https://duunitori.fi/tyopaikat/tyo/<slug>`

### Note:
**This solution fails because of Duunitori's restriction**



In [7]:
#@title <font color=#1B7192> 4. FAILED - NOT IN USE ### Scraper: Duunitori – Click to Run </font>  { display-mode: "form" }

from playwright.async_api import async_playwright, TimeoutError as PWTimeout

async def _duunitori_async(keywords):
    BASE = "https://duunitori.fi/tyopaikat"
    jobs = []
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, args=BROWSER_ARGS)
        context = await browser.new_context(
            user_agent=HEADERS["User-Agent"], locale="fi-FI"
        )
        page = await context.new_page()

        for kw in keywords:
            pg = 1
            while True:
                url = f"{BASE}?haku={requests.utils.quote(kw)}&sivu={pg}"
                print(f"  Duunitori | '{kw}' | page {pg}")
                try:
                    await page.goto(url, timeout=25000, wait_until="domcontentloaded")
                    await page.wait_for_selector(
                        "a.job-box__hover, [class*='job-box']", timeout=15000
                    )
                except PWTimeout:
                    print(f"    No cards loaded for '{kw}' page {pg} — stopping.")
                    break

                soup  = BeautifulSoup(await page.content(), "html.parser")
                cards = soup.select("a.job-box__hover") or soup.select("[class*='job-box__hover']")
                if not cards:
                    break

                for card in cards:
                    title_el    = card.select_one(".job-box__title, h3, h2, [class*='title']")
                    company_el  = card.select_one(".job-box__company, [class*='company']")
                    location_el = card.select_one(".job-box__location, [class*='location']")
                    date_el     = card.select_one(".job-box__date, [class*='date'], time")
                    href        = card.get("href", "")
                    title       = title_el.get_text(strip=True) if title_el else ""
                    if not title:
                        continue
                    jobs.append({
                        "source":   "Duunitori",
                        "keyword":  kw,
                        "title":    title,
                        "company":  company_el.get_text(strip=True)  if company_el  else "",
                        "location": location_el.get_text(strip=True) if location_el else "",
                        "posted":   date_el.get_text(strip=True)     if date_el     else "",
                        "deadline": "",
                        "url":      ("https://duunitori.fi" + href) if href.startswith("/") else href,
                    })

                # Pagination
                nxt = soup.select_one(
                    "a[rel='next'], .pagination__next a, "
                    "a[aria-label*='Seuraava'], a[aria-label*='Next']"
                )
                if not nxt:
                    break
                pg += 1
                await asyncio.sleep(1.5)

        await browser.close()
    return jobs

def scrape_duunitori(keywords):
    return asyncio.get_event_loop().run_until_complete(_duunitori_async(keywords))

print("🔍 Scraping Duunitori (Playwright async – ~1–2 min)...")
duunitori_jobs = scrape_duunitori(KEYWORDS)
all_jobs.extend(duunitori_jobs)
print(f"  → Duunitori: {len(duunitori_jobs)} listings found")

🔍 Scraping Duunitori (Playwright async – ~1–2 min)...
  Duunitori | 'arkkitehti' | page 1
    No cards loaded for 'arkkitehti' page 1 — stopping.
  Duunitori | 'projektiarkkitehti' | page 1
    No cards loaded for 'projektiarkkitehti' page 1 — stopping.
  Duunitori | 'BIM' | page 1
    No cards loaded for 'BIM' page 1 — stopping.
  Duunitori | 'architect' | page 1
    No cards loaded for 'architect' page 1 — stopping.
  → Duunitori: 0 listings found


In [20]:
#@title <font color=#1B7192> 4. Scraper: Duunitori Debug – Click to Run </font>  { display-mode: "form" }

import urllib.parse

# ── Debug mode ───────────────────────────────────────────────────────────────
# Set to True to print the raw HTML Google returns (so you can inspect the
# structure and fix selectors if needed). Set to False for normal scraping.
DEBUG = True
# ─────────────────────────────────────────────────────────────────────────────

def scrape_duunitori_via_google(keywords):
    jobs = []
    seen_urls = set()

    for kw in keywords:
        for start in range(0, 50, 10):
            query = f'site:duunitori.fi/tyopaikat/tyo/ "{kw}"'
            params = {
                "q": query,
                "num": 10,
                "start": start,
                "hl": "fi",
                "gl": "fi",
            }
            print(f"\n{'='*60}")
            print(f"  Duunitori (Google) | '{kw}' | results {start}–{start+10}")
            print(f"{'='*60}")

            r = get_html("https://www.google.com/search", params=params)
            if r is None:
                print("  ⚠️  Request failed.")
                break

            # ── DEBUG: print raw HTML so you can see what Google returned ──
            if DEBUG:
                print(f"\n--- HTTP status: {r.status_code} ---")
                print(f"--- Response length: {len(r.text)} chars ---")
                print("\n--- RAW HTML (first 3000 chars) ---")
                print(r.text[:3000])
                print("\n--- RAW HTML (last 1000 chars) ---")
                print(r.text[-1000:])

            soup = BeautifulSoup(r.text, "html.parser")

            # ── DEBUG: print every <a> href so we can see what links exist ──
            if DEBUG:
                print("\n--- ALL <a href> tags in response ---")
                all_anchors = soup.find_all("a", href=True)
                print(f"Total <a> tags found: {len(all_anchors)}")
                for a in all_anchors:
                    href = a.get("href", "")
                    text = a.get_text(strip=True)[:60]
                    print(f"  href={href[:80]!r}  text={text!r}")

                print("\n--- ALL <h3> tags in response ---")
                for h3 in soup.find_all("h3"):
                    print(f"  {h3.get_text(strip=True)!r}")

                print("\n--- Checking for known Google result selectors ---")
                for sel in ["div.g", "div.tF2Cxc", "[data-sokoban-feature]",
                            ".VwiC3b", ".IsZvec", "div[data-hveid]",
                            "div[data-async-context]", "#search", "#rso"]:
                    found = soup.select(sel)
                    print(f"  {sel!r}: {len(found)} elements")

            # ── Normal parsing ───────────────────────────────────────────────
            results = soup.select("div.g, div[data-sokoban-feature], div.tF2Cxc")
            if not results:
                results = soup.select("a[href*='duunitori.fi/tyopaikat/tyo/']")

            found_any = False
            for el in results:
                if el.name == "a":
                    link = el
                else:
                    link = el.select_one("a[href*='duunitori.fi/tyopaikat/tyo/']")
                if not link:
                    continue

                href = link.get("href", "")
                if href.startswith("/url?"):
                    href = urllib.parse.parse_qs(
                        urllib.parse.urlparse(href).query
                    ).get("q", [""])[0]
                if "duunitori.fi/tyopaikat/tyo/" not in href:
                    continue
                if href in seen_urls:
                    continue
                seen_urls.add(href)
                found_any = True

                if el.name != "a":
                    title_el   = el.select_one("h3")
                    snippet_el = el.select_one(".VwiC3b, .s3v9rd, [data-sncf], .IsZvec")
                else:
                    title_el   = None
                    snippet_el = None

                title   = title_el.get_text(strip=True)   if title_el   else link.get_text(strip=True)
                snippet = snippet_el.get_text(" ", strip=True) if snippet_el else ""

                if DEBUG:
                    print(f"\n  ✅ MATCHED JOB:")
                    print(f"     title:   {title!r}")
                    print(f"     snippet: {snippet!r}")
                    print(f"     url:     {href!r}")

                parts    = [p.strip() for p in re.split(r"[–—·|]", snippet) if p.strip()]
                company  = parts[0] if len(parts) > 0 else ""
                location = parts[1] if len(parts) > 1 else ""
                posted   = ""
                m_date   = re.search(r"Julkaistu\s+([\d\.]+)", snippet)
                if m_date:
                    posted = "Julkaistu " + m_date.group(1)

                title = re.sub(r"\s*[-–]\s*Duunitori\s*$",  "", title, flags=re.IGNORECASE).strip()
                title = re.sub(r"\s*[-–]\s*Työpaikat.*$",   "", title, flags=re.IGNORECASE).strip()
                if not title:
                    continue

                jobs.append({
                    "source":   "Duunitori",
                    "keyword":  kw,
                    "title":    title,
                    "company":  company,
                    "location": location,
                    "posted":   posted,
                    "deadline": "",
                    "url":      href,
                })

            if not found_any:
                print(f"\n  ℹ️  No Duunitori job links found in this page — stopping '{kw}'.")
                break

            time.sleep(2)

    print(f"\n{'='*60}")
    print(f"  → Duunitori: {len(jobs)} listings found")
    print(f"{'='*60}")
    return jobs


print("🔍 Scraping Duunitori (via Google Search)...")
duunitori_jobs = scrape_duunitori_via_google(KEYWORDS)
all_jobs.extend(duunitori_jobs)

🔍 Scraping Duunitori (via Google Search)...

  Duunitori (Google) | 'arkkitehti' | results 0–10
  ⚠️  Attempt 1/3 failed: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://www.google.com/search%3Fq%3Dsite%253Aduunitori.fi%252Ftyopaikat%252Ftyo%252F%2B%2522arkkitehti%2522%26num%3D10%26start%3D0%26hl%3Dfi%26gl%3Dfi&hl=fi&q=EgSC0_I3GObfsdAGIjDFdPSI872POb8COrT8x7wnpxAeJz8GAoL_qWK_wabL6T2rZwhqJDIXWD2fTLvg4eYyAnJSWgFD
  ⚠️  Attempt 2/3 failed: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://www.google.com/search%3Fq%3Dsite%253Aduunitori.fi%252Ftyopaikat%252Ftyo%252F%2B%2522arkkitehti%2522%26num%3D10%26start%3D0%26hl%3Dfi%26gl%3Dfi&hl=fi&q=EgSC0_I3GOjfsdAGIjA_m3lx9gS7SHwfAKDik6zF3bzqZJOmc2pwqMhXjlrPD062FlbPW-Ht-KZcFt1q3ygyAnJSWgFD
  ⚠️  Attempt 3/3 failed: 429 Client Error: Too Many Requests for url: https://www.google.com/sorry/index?continue=https://www.google.com/search%3Fq%3Dsite%253Aduunitori.

## Cell 5 — Scraper: Työmarkkinatori

> Search URL: `https://tyomarkkinatori.fi/henkiloasiakkaat/avoimet-tyopaikat?q=arkkitehti`  
> Job URL: `https://tyomarkkinatori.fi/henkiloasiakkaat/avoimet-tyopaikat/<uuid>/fi`

In [10]:
#@title <font color=#1B7192> 5. Scraper: Työmarkkinatori – Click to Run </font>  { display-mode: "form" }

from playwright.async_api import async_playwright, TimeoutError as PWTimeout

async def _tmt_async(keywords):
    BASE = "https://tyomarkkinatori.fi/henkiloasiakkaat/avoimet-tyopaikat"
    jobs = []
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, args=BROWSER_ARGS)
        context = await browser.new_context(
            user_agent=HEADERS["User-Agent"], locale="fi-FI"
        )
        page = await context.new_page()
        cookie_dismissed = False

        for kw in keywords:
            pg = 0
            while True:
                url = f"{BASE}?q={requests.utils.quote(kw)}&p={pg}&ps=30"
                print(f"  Työmarkkinatori | '{kw}' | page {pg}")
                try:
                    await page.goto(url, timeout=25000, wait_until="domcontentloaded")
                except PWTimeout:
                    print(f"    Page load timeout — stopping '{kw}'.")
                    break

                # Dismiss cookie banner once
                if not cookie_dismissed:
                    try:
                        await page.wait_for_selector(
                            "button:has-text('Hyväksy'), button:has-text('Accept'), "
                            "#onetrust-accept-btn-handler",
                            timeout=5000
                        )
                        await page.click(
                            "button:has-text('Hyväksy'), button:has-text('Accept'), "
                            "#onetrust-accept-btn-handler"
                        )
                        cookie_dismissed = True
                        await asyncio.sleep(1)
                    except PWTimeout:
                        cookie_dismissed = True

                # Wait for job links to appear in the rendered DOM
                try:
                    await page.wait_for_selector(
                        "a[href*='avoimet-tyopaikat/']", timeout=20000
                    )
                except PWTimeout:
                    print(f"    No results rendered for '{kw}' page {pg}.")
                    break

                await asyncio.sleep(2)   # let lazy content finish loading
                soup = BeautifulSoup(await page.content(), "html.parser")

                # Collect job detail links (UUID pattern: /avoimet-tyopaikat/<long-id>/fi)
                seen, job_links = set(), []
                for a in soup.select("a[href*='avoimet-tyopaikat/']"):
                    href = a.get("href", "")
                    if re.search(r'avoimet-tyopaikat/[\w\-]{20,}', href) and href not in seen:
                        seen.add(href)
                        job_links.append(a)

                if not job_links:
                    print(f"    No job links found for '{kw}' page {pg}.")
                    break

                for a in job_links:
                    parent      = a.find_parent(["li", "article", "div"])
                    title_el    = (parent.select_one("h2, h3, [class*='title'], [class*='Title']") if parent else None) or a
                    company_el  = parent.select_one("[class*='company'], [class*='Company'], [class*='employer']") if parent else None
                    location_el = parent.select_one("[class*='location'], [class*='Location'], [class*='municipality']") if parent else None
                    date_el     = parent.select_one("time, [class*='date'], [class*='Date']") if parent else None

                    title = title_el.get_text(strip=True) if title_el else ""
                    if title.lower() in ("lue lisää", "read more", ""):
                        title = a.get_text(strip=True)
                    if not title:
                        continue

                    href    = a.get("href", "")
                    job_url = ("https://tyomarkkinatori.fi" + href) if href.startswith("/") else href
                    jobs.append({
                        "source":   "Työmarkkinatori",
                        "keyword":  kw,
                        "title":    title,
                        "company":  company_el.get_text(strip=True)  if company_el  else "",
                        "location": location_el.get_text(strip=True) if location_el else "",
                        "posted":   date_el.get_text(strip=True)     if date_el     else "",
                        "deadline": "",
                        "url":      job_url,
                    })

                # Next page?
                try:
                    await page.wait_for_selector(
                        "button[aria-label='Next page'], button[aria-label='Seuraava sivu'], "
                        "button:has-text('Seuraava')",
                        timeout=3000
                    )
                    is_disabled = await page.evaluate(
                        """() => {
                            const b = document.querySelector(
                                'button[aria-label="Next page"], button[aria-label="Seuraava sivu"]'
                            );
                            return b ? b.disabled : true;
                        }"""
                    )
                    if is_disabled:
                        break
                    pg += 1
                    await asyncio.sleep(2)
                except PWTimeout:
                    break

        await browser.close()
    return jobs

def scrape_tyomarkkinatori(keywords):
    return asyncio.get_event_loop().run_until_complete(_tmt_async(keywords))

print("🔍 Scraping Työmarkkinatori (Playwright async – ~2–4 min)...")
tmt_jobs = scrape_tyomarkkinatori(KEYWORDS)
all_jobs.extend(tmt_jobs)
print(f"  → Työmarkkinatori: {len(tmt_jobs)} listings found")

🔍 Scraping Työmarkkinatori (Playwright async – ~2–4 min)...
  Työmarkkinatori | 'arkkitehti' | page 0
  Työmarkkinatori | 'projektiarkkitehti' | page 0
    No results rendered for 'projektiarkkitehti' page 0.
  Työmarkkinatori | 'BIM' | page 0
  Työmarkkinatori | 'architect' | page 0
  → Työmarkkinatori: 63 listings found


## Cell 6 — Scraper: Jobly

In [11]:
#@title <font color=#1B7192> 6. Scraper: Jobly – Click to Run </font>  { display-mode: "form" }


def scrape_jobly(keywords):
    BASE = "https://www.jobly.fi/en/jobs"
    jobs = []
    for kw in keywords:
        pg = 0
        while True:
            print(f"  Jobly | '{kw}' | page {pg}")
            r = get_html(BASE, params={"search": kw, "page": pg})
            if r is None:
                break
            soup  = BeautifulSoup(r.text, "html.parser")
            cards = soup.select(".job-listing, article, [class*='views-row'], [class*='job-item']")
            if not cards:
                break
            for card in cards:
                title_el    = card.select_one("h2 a, h3 a, .views-field-title a, a.listing-title")
                company_el  = card.select_one(".company-name, [class*='company'], .field-name-field-company")
                location_el = card.select_one(".location, [class*='location'], [class*='city'], .field-name-field-location")
                date_el     = card.select_one("time, .date, [class*='date'], [class*='published']")
                link_el     = title_el or card.select_one("a")
                href        = link_el.get("href", "") if link_el else ""
                title       = title_el.get_text(strip=True) if title_el else ""
                if not title:
                    continue
                jobs.append({
                    "source":   "Jobly",
                    "keyword":  kw,
                    "title":    title,
                    "company":  company_el.get_text(strip=True)  if company_el  else "",
                    "location": location_el.get_text(strip=True) if location_el else "",
                    "posted":   date_el.get_text(strip=True)     if date_el     else "",
                    "deadline": "",
                    "url":      ("https://www.jobly.fi" + href) if href.startswith("/") else href,
                })
            if not soup.select_one("a[title='Go to next page'], .pager__item--next a, li.next a"):
                break
            pg += 1
            time.sleep(1.5)
    print(f"  → Jobly: {len(jobs)} listings found")
    return jobs

print("🔍 Scraping Jobly...")
jobly_jobs = scrape_jobly(KEYWORDS)
all_jobs.extend(jobly_jobs)

🔍 Scraping Jobly...
  Jobly | 'arkkitehti' | page 0
  Jobly | 'arkkitehti' | page 1
  Jobly | 'arkkitehti' | page 2
  Jobly | 'projektiarkkitehti' | page 0
  Jobly | 'BIM' | page 0
  Jobly | 'architect' | page 0
  Jobly | 'architect' | page 1
  Jobly | 'architect' | page 2
  Jobly | 'architect' | page 3
  → Jobly: 258 listings found


## Cell 7 — Scraper: SAFA

> **Fix:** SAFA's jobs page has **no search/filter functionality** – all listings are shown on a single page as plain links.
> The scraper fetches the page once and returns all listings (they are all architecture jobs by nature).
> Keyword is stored as `'all'` since SAFA only posts architecture positions.
>
> Page structure confirmed:
> ```
> <h2><a href="/tyopaikat/...">Job title</a></h2>
> Työnantaja: Company name
> Haku loppuu: DD.MM.YYYY
> ```

In [12]:
#@title <font color=#1B7192> 7. Scraper: SAFA – Click to Run </font>  { display-mode: "form" }

def scrape_safa():
    URL  = "https://www.safa.fi/koulutus-ja-ura/tyopaikat/"
    jobs = []
    print(f"  SAFA | fetching {URL}")
    r = get_html(URL)
    if r is None:
        print("  ⚠️  SAFA: could not fetch page")
        return jobs
    soup = BeautifulSoup(r.text, "html.parser")

    # Job detail links: /tyopaikat/<slug>/  (bare <a> tags, NOT inside <h2>)
    all_links = soup.find_all("a", href=re.compile(r"/tyopaikat/[^/]+/?$"))
    seen_urls = set()
    for link in all_links:
        href  = link.get("href", "")
        title = link.get_text(strip=True)
        if (
            not title
            or title.lower() in ("lue lisää", "read more", "työpaikat", "avoimet työpaikat")
            or "/koulutus-ja-ura/tyopaikat" in href
            or href in seen_urls
        ):
            continue
        seen_urls.add(href)

        # Collect sibling text to find company & deadline
        sibling_text = ""
        for sib in link.next_siblings:
            s = sib.get_text(" ", strip=True) if hasattr(sib, "get_text") else str(sib).strip()
            if s:
                sibling_text += " " + s
            if len(sibling_text) > 300:
                break

        m_co   = re.search(r"Ty[öo]nantaja[:\s]+([^\n\r\[\]]+)", sibling_text, re.IGNORECASE)
        m_date = re.search(r"Haku loppuu[:\s]+([\d\.]+)",          sibling_text, re.IGNORECASE)
        jobs.append({
            "source":   "SAFA",
            "keyword":  "all",
            "title":    title,
            "company":  m_co.group(1).strip().rstrip(",") if m_co   else "",
            "location": "",
            "posted":   "",
            "deadline": m_date.group(1).strip()           if m_date else "",
            "url":      ("https://www.safa.fi" + href) if href.startswith("/") else href,
        })

    print(f"  → SAFA: {len(jobs)} listings found")
    return jobs

print("🔍 Scraping SAFA...")
safa_jobs = scrape_safa()
all_jobs.extend(safa_jobs)

🔍 Scraping SAFA...
  SAFA | fetching https://www.safa.fi/koulutus-ja-ura/tyopaikat/
  → SAFA: 3 listings found


## Cell 8 — Combine, deduplicate & clean

In [17]:
#@title <font color=#1B7192> 8. Combine & clean – Click to Run </font>  { display-mode: "form" }


if not all_jobs:
    print("⚠️  all_jobs is empty. Re-run Cell 3 to reset, then cells 4–7.")
else:
    df = pd.DataFrame(all_jobs)
    for col in ["source", "keyword", "title", "company", "location", "posted", "deadline", "url"]:
        if col not in df.columns:
            df[col] = ""
    df = df.fillna("")
    df = df[df["title"].str.strip() != ""]
    df = df.drop_duplicates(subset=["url"])
    df = df.drop_duplicates(subset=["title", "company"])
    for col in ["title", "company", "location", "posted", "deadline"]:
        df[col] = df[col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df["posted"]     = df["posted"].str.rstrip(",")
    df["scraped_at"] = datetime.now().strftime("%Y-%m-%d %H:%M")
    df = df[["source", "keyword", "title", "company", "location",
              "posted", "deadline", "url", "scraped_at"]].reset_index(drop=True)

    print(f"\n✅ Total unique listings: {len(df)}")
    print("\nBreakdown by source:")
    print(df.groupby("source")["title"].count().rename("count").to_string())
    display(df.head(12))


✅ Total unique listings: 137

Breakdown by source:
source
Duunitori           1
Jobly              77
SAFA                3
Työmarkkinatori    56


,source,keyword,title,company,location,posted,deadline,url,scraped_at
0,Työmarkkinatori,arkkitehti,Salauksen ja avainhallinnan regulaatioasiantun...,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
1,Työmarkkinatori,arkkitehti,Vesihuollon laitossuunnittelija,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
2,Työmarkkinatori,arkkitehti,Vesihuollon ja hulevesien Design Lead,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
3,Työmarkkinatori,arkkitehti,Rakennuttamis- ja projektinjohtotehtävät ja ra...,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
4,Työmarkkinatori,arkkitehti,Arkkitehtiosaston tiiminvetäjä,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
5,Työmarkkinatori,arkkitehti,"Data Coordinator IT-hankkeeseen, UPM / Helsink...",,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
6,Työmarkkinatori,arkkitehti,Sähkösuunnittelun tiimipäällikkö,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
7,Työmarkkinatori,arkkitehti,"Tiimipäällikkö, vaativa lujuuslaskenta",,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
8,Työmarkkinatori,arkkitehti,Division Controller,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34
9,Työmarkkinatori,arkkitehti,Myyntiedustaja / Sales Manager valaistusratkaisut,,,,,https://tyomarkkinatori.fi/henkiloasiakkaat/av...,2026-05-19 13:34


## Cell 9 — Export to Excel & download

In [16]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1RdJdK_A_5xBfu4Zs2SxZwjxXuf9enMR697NCfYFFVpM/edit#gid=0


In [ ]:
#@title <font color=#1B7192> 9. Export & download – Click to Run </font>  { display-mode: "form" }


from google.colab import files
filename = f"job_listings_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="All Jobs", index=False)
    for source, group in df.groupby("source"):
        group.to_excel(writer, sheet_name=source[:31], index=False)
    for sheet in writer.sheets.values():
        for col in sheet.columns:
            w = max((len(str(c.value)) for c in col if c.value), default=10)
            sheet.column_dimensions[col[0].column_letter].width = min(w + 4, 60)
print(f"📁 Saved: {filename}")
files.download(filename)

📁 Saved: job_listings_20260519_0956.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>